# Evaluation: Gemma vs Vault Gemma - Geometric Representation Comparison

This notebook compares the geometric representations of categorical hierarchies between standard Gemma and Vault Gemma (with differential privacy).

We evaluate two key metrics:
1. **Intra-Class Variance** (based on Theorem 4 of the paper)
2. **Hierarchical Orthogonality**

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plotting style
sns.set_theme(context="paper", style="whitegrid", palette="colorblind", font_scale=1.5)

print("Loading results from previous analyses...")

## 1. Load Results from Both Models

In [ ]:
# Load Gemma results
with open('gemma_results.pkl', 'rb') as f:
    gemma_results = pickle.load(f)

# Load Vault Gemma results
with open('vault_gemma_results.pkl', 'rb') as f:
    vault_results = pickle.load(f)

print(f"Gemma Model: {gemma_results['model_name']}")
print(f"Vault Gemma Model: {vault_results['model_name']}")
print(f"\nCategories analyzed: {gemma_results['categories']}")

## 2. Metric 1: Intra-Class Variance Comparison

### Theory (Theorem 4 of the paper):
For a perfect concept, the projection of every word in that category onto the concept vector should be a constant value `b`. In reality, it forms a distribution.

### Hypothesis:
**If differential privacy (DP) adds noise, this distribution should widen.**

If `σ_vault >> σ_gemma`, the hypothesis is TRUE. The "walls" of the polytope are blurry; the DP model is less certain about exactly which words belong to a category.

### Visualization:
Plot a histogram of projection values. The DP model's histogram should look shorter and wider ("pancaked") compared to the sharp peak of the standard model.

In [ ]:
# Extract variance statistics
categories = gemma_results['categories']

gemma_stds = []
vault_stds = []
gemma_vars = []
vault_vars = []

for cat in categories:
    if cat in gemma_results['projection_stats'] and cat in vault_results['projection_stats']:
        gemma_stds.append(gemma_results['projection_stats'][cat]['std'])
        vault_stds.append(vault_results['projection_stats'][cat]['std'])
        gemma_vars.append(gemma_results['projection_stats'][cat]['variance'])
        vault_vars.append(vault_results['projection_stats'][cat]['variance'])

# Compute statistics
mean_gemma_std = np.mean(gemma_stds)
mean_vault_std = np.mean(vault_stds)
ratio_std = mean_vault_std / mean_gemma_std

print("=" * 70)
print("INTRA-CLASS VARIANCE COMPARISON")
print("=" * 70)
print(f"\n{'Category':<15} {'σ_gemma':<12} {'σ_vault':<12} {'Ratio (σ_v/σ_g)':<15}")
print("-" * 70)
for cat, g_std, v_std in zip(categories, gemma_stds, vault_stds):
    ratio = v_std / g_std if g_std > 0 else float('inf')
    print(f"{cat:<15} {g_std:<12.6f} {v_std:<12.6f} {ratio:<15.4f}")

print("-" * 70)
print(f"{'AVERAGE':<15} {mean_gemma_std:<12.6f} {mean_vault_std:<12.6f} {ratio_std:<15.4f}")
print("=" * 70)

if ratio_std > 1.5:
    print(f"\n✓ HYPOTHESIS SUPPORTED: σ_vault >> σ_gemma (ratio = {ratio_std:.2f})")
    print("  The DP model has significantly wider projection distributions.")
    print("  The 'walls' of the polytope are blurry - less certain category boundaries.")
elif ratio_std > 1.1:
    print(f"\n~ HYPOTHESIS PARTIALLY SUPPORTED: σ_vault > σ_gemma (ratio = {ratio_std:.2f})")
    print("  The DP model has moderately wider projection distributions.")
else:
    print(f"\n✗ HYPOTHESIS NOT SUPPORTED: σ_vault ≈ σ_gemma (ratio = {ratio_std:.2f})")
    print("  The DP model does not show significantly wider distributions.")

### Visualization: Standard Deviation Comparison

In [ ]:
# Bar plot comparing standard deviations
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
x = np.arange(len(categories))
width = 0.35

ax.bar(x - width/2, gemma_stds, width, label='Gemma (Standard)', alpha=0.8, color='blue')
ax.bar(x + width/2, vault_stds, width, label='Vault Gemma (DP)', alpha=0.8, color='green')

ax.set_xlabel('Category')
ax.set_ylabel('Standard Deviation (σ)')
ax.set_title('Intra-Class Variance: Projection Standard Deviations\n(Higher = More Uncertain Category Boundaries)')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_intra_class_variance.png', dpi=300, bbox_inches='tight')
plt.show()

### Visualization: Projection Distribution Histograms (Side-by-Side)

In [ ]:
# Create side-by-side histograms for each category
fig, axes = plt.subplots(len(categories), 2, figsize=(14, 3*len(categories)))

for idx, category in enumerate(categories):
    if category in gemma_results['projection_stats'] and category in vault_results['projection_stats']:
        # Gemma histogram
        gemma_proj = gemma_results['projection_stats'][category]['projections']
        axes[idx, 0].hist(gemma_proj, bins=30, alpha=0.7, color='blue', edgecolor='black')
        axes[idx, 0].axvline(gemma_results['projection_stats'][category]['mean'], 
                            color='red', linestyle='--', linewidth=2, label='Mean')
        axes[idx, 0].set_title(f"Gemma: {category}\nσ={gemma_results['projection_stats'][category]['std']:.4f}")
        axes[idx, 0].set_xlabel('Projection Value')
        axes[idx, 0].set_ylabel('Frequency')
        
        # Vault Gemma histogram
        vault_proj = vault_results['projection_stats'][category]['projections']
        axes[idx, 1].hist(vault_proj, bins=30, alpha=0.7, color='green', edgecolor='black')
        axes[idx, 1].axvline(vault_results['projection_stats'][category]['mean'], 
                            color='red', linestyle='--', linewidth=2, label='Mean')
        axes[idx, 1].set_title(f"Vault Gemma: {category}\nσ={vault_results['projection_stats'][category]['std']:.4f}")
        axes[idx, 1].set_xlabel('Projection Value')
        axes[idx, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('comparison_projection_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nNote: Look for 'pancaked' distributions in Vault Gemma (wider, shorter)")
print("compared to sharper peaks in standard Gemma.")

### Statistical Test: Variance Ratio

In [ ]:
# Perform paired t-test on variances
t_stat, p_value = stats.ttest_rel(vault_vars, gemma_vars)

print("\nPaired t-test for variance differences:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.6f}")

if p_value < 0.05:
    if t_stat > 0:
        print("  Result: Vault Gemma has SIGNIFICANTLY HIGHER variance (p < 0.05)")
    else:
        print("  Result: Vault Gemma has SIGNIFICANTLY LOWER variance (p < 0.05)")
else:
    print("  Result: No significant difference in variance (p >= 0.05)")

## 3. Metric 2: Hierarchical Orthogonality Comparison

### Theory:
In a hierarchical relationship (e.g., Animal → Mammal), the difference vector `d = l_child - l_parent` should be orthogonal to the parent vector `l_parent`.

### Hypothesis:
**Hierarchical orthogonality should be preserved (or possibly stronger) in Vault Gemma.**

### Measurement:
- Compute `cos(l_parent, d)` for each parent-child relationship
- Target: Close to 0 for perfect orthogonality
- Compare absolute values: |cos_vault| vs |cos_gemma|

In [ ]:
# Extract hierarchical orthogonality statistics
gemma_cos_parent_diff = []
vault_cos_parent_diff = []
gemma_cos_parent_child = []
vault_cos_parent_child = []

for cat in categories:
    if cat in gemma_results['hierarchical_stats'] and cat in vault_results['hierarchical_stats']:
        gemma_cos_parent_diff.append(gemma_results['hierarchical_stats'][cat]['cos_parent_diff'])
        vault_cos_parent_diff.append(vault_results['hierarchical_stats'][cat]['cos_parent_diff'])
        gemma_cos_parent_child.append(gemma_results['hierarchical_stats'][cat]['cos_parent_child'])
        vault_cos_parent_child.append(vault_results['hierarchical_stats'][cat]['cos_parent_child'])

# Compute mean absolute cosine similarities
mean_gemma_ortho = np.mean(np.abs(gemma_cos_parent_diff))
mean_vault_ortho = np.mean(np.abs(vault_cos_parent_diff))

print("=" * 80)
print("HIERARCHICAL ORTHOGONALITY COMPARISON")
print("=" * 80)
print(f"\n{'Category':<15} {'cos_gemma(l_p, d)':<20} {'cos_vault(l_p, d)':<20} {'|Δ|':<15}")
print("-" * 80)
for cat, g_cos, v_cos in zip(categories, gemma_cos_parent_diff, vault_cos_parent_diff):
    delta = abs(v_cos) - abs(g_cos)
    print(f"{cat:<15} {g_cos:<20.6f} {v_cos:<20.6f} {delta:<15.6f}")

print("-" * 80)
print(f"{'AVERAGE |cos|':<15} {mean_gemma_ortho:<20.6f} {mean_vault_ortho:<20.6f}")
print("=" * 80)

print("\nInterpretation:")
print("  - Values closer to 0 indicate better hierarchical orthogonality")
print("  - Negative Δ means Vault Gemma has BETTER orthogonality (closer to 0)")
print("  - Positive Δ means standard Gemma has BETTER orthogonality")

if mean_vault_ortho < mean_gemma_ortho:
    improvement = (mean_gemma_ortho - mean_vault_ortho) / mean_gemma_ortho * 100
    print(f"\n✓ Vault Gemma has BETTER hierarchical orthogonality ({improvement:.1f}% improvement)")
elif mean_vault_ortho > mean_gemma_ortho * 1.1:
    degradation = (mean_vault_ortho - mean_gemma_ortho) / mean_gemma_ortho * 100
    print(f"\n✗ Vault Gemma has WORSE hierarchical orthogonality ({degradation:.1f}% degradation)")
else:
    print(f"\n~ Hierarchical orthogonality is COMPARABLE between models")

### Visualization: Hierarchical Orthogonality Comparison

In [ ]:
# Bar plot comparing hierarchical orthogonality
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(categories))
width = 0.35

# Plot 1: cos(l_parent, d)
axes[0].bar(x - width/2, gemma_cos_parent_diff, width, label='Gemma (Standard)', alpha=0.8, color='blue')
axes[0].bar(x + width/2, vault_cos_parent_diff, width, label='Vault Gemma (DP)', alpha=0.8, color='green')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Perfect Orthogonality')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('cos(l_parent, d)')
axes[0].set_title('Hierarchical Orthogonality: cos(l_parent, d)\n(Closer to 0 = Better)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Absolute values
axes[1].bar(x - width/2, np.abs(gemma_cos_parent_diff), width, label='Gemma (Standard)', alpha=0.8, color='blue')
axes[1].bar(x + width/2, np.abs(vault_cos_parent_diff), width, label='Vault Gemma (DP)', alpha=0.8, color='green')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('|cos(l_parent, d)|')
axes[1].set_title('Hierarchical Orthogonality: |cos(l_parent, d)|\n(Lower = Better)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_hierarchical_orthogonality.png', dpi=300, bbox_inches='tight')
plt.show()

### Statistical Test: Orthogonality Preservation

In [ ]:
# Perform paired t-test on absolute cosine similarities
abs_gemma = np.abs(gemma_cos_parent_diff)
abs_vault = np.abs(vault_cos_parent_diff)

t_stat, p_value = stats.ttest_rel(abs_vault, abs_gemma)

print("\nPaired t-test for hierarchical orthogonality:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.6f}")

if p_value < 0.05:
    if t_stat < 0:
        print("  Result: Vault Gemma has SIGNIFICANTLY BETTER orthogonality (p < 0.05)")
    else:
        print("  Result: Vault Gemma has SIGNIFICANTLY WORSE orthogonality (p < 0.05)")
else:
    print("  Result: No significant difference in orthogonality (p >= 0.05)")

## 4. Combined Visualization: Summary Comparison

In [ ]:
# Create a comprehensive summary figure
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Plot 1: Standard Deviation Comparison
ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(len(categories))
width = 0.35
ax1.bar(x - width/2, gemma_stds, width, label='Gemma', alpha=0.8, color='blue')
ax1.bar(x + width/2, vault_stds, width, label='Vault Gemma', alpha=0.8, color='green')
ax1.set_xlabel('Category')
ax1.set_ylabel('Standard Deviation (σ)')
ax1.set_title('(A) Intra-Class Variance')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Variance Ratio
ax2 = fig.add_subplot(gs[0, 1])
ratios = [v/g if g > 0 else 0 for v, g in zip(vault_stds, gemma_stds)]
colors_ratio = ['green' if r > 1 else 'blue' for r in ratios]
ax2.bar(x, ratios, alpha=0.8, color=colors_ratio)
ax2.axhline(y=1, color='red', linestyle='--', linewidth=2, label='Equal Variance')
ax2.set_xlabel('Category')
ax2.set_ylabel('Ratio (σ_vault / σ_gemma)')
ax2.set_title('(B) Variance Ratio\n(>1 = Vault has higher variance)')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Hierarchical Orthogonality (Absolute)
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(x - width/2, abs_gemma, width, label='Gemma', alpha=0.8, color='blue')
ax3.bar(x + width/2, abs_vault, width, label='Vault Gemma', alpha=0.8, color='green')
ax3.set_xlabel('Category')
ax3.set_ylabel('|cos(l_parent, d)|')
ax3.set_title('(C) Hierarchical Orthogonality\n(Lower = Better)')
ax3.set_xticks(x)
ax3.set_xticklabels(categories, rotation=45, ha='right')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Orthogonality Improvement
ax4 = fig.add_subplot(gs[1, 1])
ortho_delta = abs_gemma - abs_vault  # Positive = Vault is better
colors_ortho = ['green' if d > 0 else 'red' for d in ortho_delta]
ax4.bar(x, ortho_delta, alpha=0.8, color=colors_ortho)
ax4.axhline(y=0, color='black', linestyle='--', linewidth=2)
ax4.set_xlabel('Category')
ax4.set_ylabel('Orthogonality Improvement\n(|cos_gemma| - |cos_vault|)')
ax4.set_title('(D) Orthogonality Change\n(>0 = Vault is better)')
ax4.set_xticks(x)
ax4.set_xticklabels(categories, rotation=45, ha='right')
ax4.grid(axis='y', alpha=0.3)

plt.suptitle('Comprehensive Comparison: Gemma vs Vault Gemma', fontsize=18, y=0.995)
plt.savefig('comparison_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Final Summary and Conclusions

In [ ]:
print("="*80)
print("FINAL EVALUATION SUMMARY")
print("="*80)

print("\n1. INTRA-CLASS VARIANCE (Theorem 4)")
print("-" * 80)
print(f"   Average σ_gemma:      {mean_gemma_std:.6f}")
print(f"   Average σ_vault:      {mean_vault_std:.6f}")
print(f"   Ratio (σ_vault/σ_gemma): {ratio_std:.4f}")

if ratio_std > 1.5:
    print("\n   ✓ CONCLUSION: Differential privacy significantly increases intra-class variance.")
    print("     The polytope 'walls' are blurrier in Vault Gemma - less certain category boundaries.")
elif ratio_std > 1.1:
    print("\n   ~ CONCLUSION: Differential privacy moderately increases intra-class variance.")
else:
    print("\n   ✗ CONCLUSION: No significant increase in variance from differential privacy.")

print("\n2. HIERARCHICAL ORTHOGONALITY")
print("-" * 80)
print(f"   Average |cos_gemma|:  {mean_gemma_ortho:.6f}")
print(f"   Average |cos_vault|:  {mean_vault_ortho:.6f}")
print(f"   Difference:           {mean_vault_ortho - mean_gemma_ortho:.6f}")

if mean_vault_ortho < mean_gemma_ortho * 0.9:
    print("\n   ✓ CONCLUSION: Vault Gemma has STRONGER hierarchical orthogonality.")
    print("     Differential privacy preserves or enhances the hierarchical structure.")
elif mean_vault_ortho > mean_gemma_ortho * 1.1:
    print("\n   ✗ CONCLUSION: Vault Gemma has WEAKER hierarchical orthogonality.")
    print("     Differential privacy degrades the hierarchical structure.")
else:
    print("\n   ~ CONCLUSION: Hierarchical orthogonality is PRESERVED.")
    print("     Differential privacy does not significantly affect hierarchical structure.")

print("\n" + "="*80)
print("OVERALL FINDINGS")
print("="*80)
print("\nDifferential privacy in Vault Gemma affects the geometric representation of")
print("categorical concepts by:")
print(f"  1. {'INCREASING' if ratio_std > 1.2 else 'MAINTAINING'} the uncertainty within category boundaries (intra-class variance)")
print(f"  2. {'PRESERVING' if abs(mean_vault_ortho - mean_gemma_ortho) < 0.01 else 'MODIFYING'} the hierarchical orthogonality structure")
print("\nThese results align with expectations from the paper's theoretical framework")
print("regarding polytope geometry and hierarchical concept encoding.")
print("="*80)

## 6. Export Results for Further Analysis

In [ ]:
# Create a comprehensive results dictionary
comparison_results = {
    'categories': categories,
    'intra_class_variance': {
        'gemma_stds': gemma_stds,
        'vault_stds': vault_stds,
        'gemma_vars': gemma_vars,
        'vault_vars': vault_vars,
        'mean_gemma_std': mean_gemma_std,
        'mean_vault_std': mean_vault_std,
        'ratio': ratio_std
    },
    'hierarchical_orthogonality': {
        'gemma_cos_parent_diff': gemma_cos_parent_diff,
        'vault_cos_parent_diff': vault_cos_parent_diff,
        'mean_gemma_ortho': mean_gemma_ortho,
        'mean_vault_ortho': mean_vault_ortho
    }
}

with open('comparison_results.pkl', 'wb') as f:
    pickle.dump(comparison_results, f)

print("Comparison results saved to 'comparison_results.pkl'")
print("\n=== Evaluation Complete ===")

## References

This analysis is based on:
- **Paper**: "THE GEOMETRY OF CATEGORICAL AND HIERARCHICAL CONCEPTS IN LARGE LANGUAGE MODELS"
- **Key Theorems**:
  - Theorem 4: Characterizes the projection distribution of words onto concept vectors
  - Hierarchical orthogonality principle: Parent-child relationships encoded as orthogonal directions

## Next Steps

1. Run analysis with different model sizes (Gemma-7b, etc.)
2. Test with different hierarchies (plants, verbs from WordNet)
3. Investigate specific categories showing largest differences
4. Analyze computational implications of increased variance